# Capstone — FlyRank Refresh Opportunity Ranking

This notebook reproduces the capstone workflow on the bundled anonymized starter dataset. The lane is a real FlyRank content problem: editors need a ranked queue of pages that still carry demand but are slipping enough to deserve a human refresh review before traffic drops further.


## 1. Question

The project asks: among FlyRank pages with visible demand, which ones deserve a human review first? The output is a ranked refresh-opportunity queue for editorial triage, not a claim about absolute search rank or algorithmic causality.


In [1]:
import json
from pathlib import Path

root = Path.cwd().resolve()
for _ in range(4):
    if (root / 'outputs').exists() and (root / 'data').exists():
        break
    root = root.parent
results = json.loads((root / 'outputs' / 'model_results.json').read_text())
print(f"Best model: {results['best_model']['name']}")
print(f"Precision@50: {results['models']['random_forest']['precision_at_50']:.3f}")
print(f"Baseline Precision@50: {results['baseline']['baseline_precision_at_50']:.3f}")


Best model: random_forest
Precision@50: 0.680
Baseline Precision@50: 0.240


## 2. Data

The analysis uses the gated FlyRank warehouse release via Hugging Face (`hf://datasets/FlyRank/internship-warehouse`). We read a mid-panel month partition (`month=2026-03`) using an `HF_TOKEN` loaded from `.env` (or environment variable), then keep feature work in a public-safe lane without exposing token values.

In [2]:
import os
import duckdb
import pandas as pd
from pathlib import Path

def load_hf_token():
    for path in [Path.cwd() / '.env', Path.cwd().parent / '.env', Path.cwd().parent.parent / '.env']:
        if path.exists():
            for line in path.read_text().splitlines():
                if line.strip().startswith('HF_TOKEN='):
                    return line.split('=', 1)[1].strip().strip('"').strip(chr(39))
    return os.environ.get('HF_TOKEN')

token = load_hf_token()
if not token:
    raise RuntimeError('HF_TOKEN required via .env or the environment.')

FACT = 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
con = duckdb.connect()
con.execute('CREATE OR REPLACE SECRET hf (TYPE HUGGINGFACE, TOKEN ?)', [token])
df = con.execute("""
SELECT *
FROM read_parquet(?)
WHERE gsc_data_available IS TRUE
LIMIT 30000
""", [FACT]).df()
print(f'Rows pulled from warehouse partition: {len(df):,}')
print(f'Columns: {len(df.columns)}')
print(df.head(3).to_string(index=False))


Rows pulled from warehouse partition: 30,000
Columns: 31
report_date          client_hash_id          content_hash_id  client_has_gsc  client_has_ga4  gsc_data_available  ga4_data_available  gsc_impressions  gsc_clicks  gsc_sum_position  gsc_avg_position  ga4_pageviews  ga4_sessions  ga4_users  ga4_engaged_sessions  ga4_total_engagement_sec  sessions_organic  sessions_direct  sessions_referral  sessions_social  sessions_paid  sessions_ai  ai_chatgpt  ai_perplexity  ai_gemini  ai_copilot  ai_claude  ai_meta  ai_other  scroll_events   month
 2026-03-01 client_73cda7b4e4f265ea content_b7e512995f79d5a6            True           False                True                <NA>               20           0                67             3.350           <NA>          <NA>       <NA>                  <NA>                      <NA>              <NA>             <NA>               <NA>             <NA>           <NA>         <NA>        <NA>           <NA>       <NA>        <NA>       <NA>     <NA> 

## 3. Methodology

We prepare a feature vector, define a declining-page label, and compare a transparent rule baseline to a learned model. Validation is client-aware so pages from the same client do not leak across train and test splits.

In [3]:
import json
from pathlib import Path

root = Path.cwd().resolve()
for _ in range(4):
    if (root / 'outputs').exists() and (root / 'data').exists():
        break
    root = root.parent
meta = json.loads((root / 'data' / 'processed' / 'feature_metadata.json').read_text())
print(json.dumps({'rows': meta['prepared_rows'], 'declining_rate': round(meta['declining_rate'], 3), 'target': meta['target_definition'], 'features': len(meta['model_numeric_features']) + len(meta['model_categorical_features'])}, indent=2))


{
  "rows": 30000,
  "declining_rate": 0.542,
  "target": "trend_direction == 'down'",
  "features": 26
}


## 4. Results (vs baseline)

This is the honest comparison on the same split. The model is selected by Precision@50, and the final comparison is against the baseline rule from the repo pipeline.

In [4]:
import json
import pandas as pd
from pathlib import Path

root = Path.cwd().resolve()
for _ in range(4):
    if (root / 'outputs').exists() and (root / 'data').exists():
        break
    root = root.parent
results = json.loads((root / 'outputs' / 'model_results.json').read_text())
metrics = []
for name, payload in results['models'].items():
    metrics.append({
        'Model': name,
        'ROC AUC': round(payload['roc_auc'], 3),
        'Avg Precision': round(payload['average_precision'], 3),
        'Precision@50': payload['precision_at_50'],
        'Recall': round(payload['recall'], 3),
        'F1': round(payload['f1'], 3),
    })
metrics.append({
    'Model': 'baseline_rules',
    'ROC AUC': round(results['baseline']['baseline_roc_auc'], 3),
    'Avg Precision': round(results['baseline']['baseline_average_precision'], 3),
    'Precision@50': results['baseline']['baseline_precision_at_50'],
    'Recall': None,
    'F1': None,
})
print(pd.DataFrame(metrics).to_string(index=False))


              Model  ROC AUC  Avg Precision  Precision@50  Recall    F1
      decision_tree    0.742          0.575          0.62   0.716 0.634
logistic_regression    0.700          0.522          0.40   0.567 0.566
      random_forest    0.747          0.610          0.68   0.741 0.638
     baseline_rules    0.627          0.468          0.24     NaN   NaN


## 5. Limitations

This project can support a review queue and editorial triage, but it cannot prove causality or claim that it predicts Google’s algorithm. It is directional, measured on historical data, and useful as a decision-support tool rather than a deterministic ranking system.

In [5]:
import pandas as pd
from pathlib import Path

root = Path.cwd().resolve()
for _ in range(4):
    if (root / 'outputs').exists() and (root / 'data').exists():
        break
    root = root.parent
queue = pd.read_csv(root / 'outputs' / 'refresh_queue.csv')
print(f'Top queue rows: {len(queue)}')
print(queue[['content_id', 'final_rank', 'best_model_probability', 'suggested_action', 'final_reason_codes']].head(5).to_string(index=False))


Top queue rows: 30000
          content_id  final_rank  best_model_probability       suggested_action                                                                                                                                                   final_reason_codes
content_1f080331fa2b           1                0.786247 refresh_and_review_ctr declining_with_demand|low_ctr_visible_page|low_engagement_visible_page|model_decline_risk|visible_model_opportunity|ctr_review_candidate|engagement_review_candidate
content_6aa43079fb0c           2                0.792117 refresh_and_review_ctr                                                         declining_with_demand|low_ctr_visible_page|model_decline_risk|visible_model_opportunity|ctr_review_candidate
content_d6570c51c9bd           3                0.850354 refresh_and_review_ctr                                                         declining_with_demand|low_ctr_visible_page|model_decline_risk|visible_model_opportunity|ctr_review_candidat

## 6. Ranked recommendations

Top-ranked pages should be reviewed by a human editor. The queue is a triage tool: inspect the page, verify the signal against business context, and decide whether to refresh, expand, or improve CTR/engagement.

In [6]:
import pandas as pd
from pathlib import Path

root = Path.cwd().resolve()
for _ in range(4):
    if (root / 'outputs').exists() and (root / 'data').exists():
        break
    root = root.parent
queue = pd.read_csv(root / 'outputs' / 'refresh_queue.csv')
print(queue.head(10)[['content_id', 'final_rank', 'best_model_probability', 'suggested_action', 'final_reason_codes']].to_string(index=False))


          content_id  final_rank  best_model_probability       suggested_action                                                                                                                                                   final_reason_codes
content_1f080331fa2b           1                0.786247 refresh_and_review_ctr declining_with_demand|low_ctr_visible_page|low_engagement_visible_page|model_decline_risk|visible_model_opportunity|ctr_review_candidate|engagement_review_candidate
content_6aa43079fb0c           2                0.792117 refresh_and_review_ctr                                                         declining_with_demand|low_ctr_visible_page|model_decline_risk|visible_model_opportunity|ctr_review_candidate
content_d6570c51c9bd           3                0.850354 refresh_and_review_ctr                                                         declining_with_demand|low_ctr_visible_page|model_decline_risk|visible_model_opportunity|ctr_review_candidate
content_e04eb9549989

## 7. Artifacts the paper embeds

The capstone report should include the key charts and model summary files generated by the repo pipeline. These are the evidence artifacts for the page and the final write-up.

In [7]:
from pathlib import Path

root = Path.cwd().resolve()
for _ in range(4):
    if (root / 'outputs').exists() and (root / 'data').exists():
        break
    root = root.parent
artifacts = sorted((root / 'outputs' / 'charts').glob('*'))
print('Chart files:')
for item in artifacts:
    print('-', item.name)
print('')
print('Summary files:')
for item in [root / 'outputs' / 'model_report.md', root / 'outputs' / 'model_results.json', root / 'outputs' / 'refresh_queue.csv']:
    print('-', item.relative_to(root))


Chart files:
- action_mix.svg
- confidence_mix.svg
- top_feature_importance.svg
- top_reason_codes.svg
- trend_distribution.svg

Summary files:
- outputs\model_report.md
- outputs\model_results.json
- outputs\refresh_queue.csv


## 8. ML-12 closing summary

### 5-minute demo outline

1. **Question:** Which FlyRank pages still have visible demand but are starting to slip, and therefore deserve a human refresh review first?
2. **Method:** Use the anonymized page dataset, define the declining-page label, compare a transparent rule baseline to a client-holdout random forest, and rank the queue by expected review value.
3. **One chart:** show the top feature importance chart to explain that the model is responding to page visibility and freshness signals, not random noise.
4. **One honest result:** the model lifts Precision@50 from 0.240 for the baseline to 0.680, a roughly 2.8x improvement at the top of the queue.
5. **Recommendation:** treat this as a triage tool—look at the top-ranked pages first, verify them with editorial context, then decide whether to refresh, expand, or improve CTR/engagement.

### Social post

FlyRank content refresh triage, in one sentence: we built a client-holdout ranking model that uses visibility, engagement, and freshness signals to surface the pages most worth a human review before traffic slips further. It is a ranked decision-support tool, not an automatic content edit.

### Employer-facing summary

I built a content-refresh prioritization model for FlyRank that ranks pages most likely to deserve editorial review. It used 30,000 anonymized page records and client-holdout validation across visibility, engagement, and freshness signals. The random forest improved Precision@50 from 0.240 for the rule baseline to 0.680, showing the model can surface the highest-value refresh candidates without replacing human judgment.


## Self-check

- [x] The notebook is filled and explains the project honestly
- [x] The pipeline artifacts exist under `outputs/`
- [x] No client-identifying info is exposed
- [x] Claims stay measured and decision-support oriented

In [8]:
from pathlib import Path

root = Path.cwd().resolve()
for _ in range(4):
    if (root / 'outputs').exists() and (root / 'data').exists():
        break
    root = root.parent
checks = [
    (root / 'outputs' / 'model_report.md').exists(),
    (root / 'outputs' / 'model_results.json').exists(),
    (root / 'outputs' / 'refresh_queue.csv').exists(),
    (root / 'outputs' / 'flyrank_refresh_model_results.pdf').exists(),
]
print('Artifact checks:', checks)
print('All required outputs present:', all(checks))


Artifact checks: [True, True, True, True]
All required outputs present: True
